In [1]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai_api_key = os.environ["OPENAI_API_KEY"]

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

In [3]:
from langchain_core.messages import HumanMessage
from langgraph.graph import END, MessageGraph


agent = MessageGraph()

In [4]:
agent.add_node("node1", llm)

In [5]:
agent.add_edge("node1", END)
# burada bu node çalıştıysa sonlandır başka node a gitme demek

In [6]:
agent.set_entry_point("node1")

In [7]:
runnable_agent = agent.compile()

In [8]:
runnable_agent.invoke(HumanMessage("What is 1 + 1?"))

[HumanMessage(content='What is 1 + 1?', id='dd2057c4-09ce-4954-95b1-ce46cd79d5ca'),
 AIMessage(content='1 + 1 equals 2.', response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 15, 'total_tokens': 24, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_dbaca60df0', 'finish_reason': 'stop', 'logprobs': None}, id='run-411cda22-af29-46d8-aa56-b1bf4f53628d-0', usage_metadata={'input_tokens': 15, 'output_tokens': 9, 'total_tokens': 24})]

In [9]:
## burada tek node lu bir agent yaptık

In [10]:
#şimdi tool ekleyelim
# çok node lu naşka bir agent yapalım

In [9]:
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode

@tool
def multiply(first_number: int, second_number: int):
    """Multiplies two numbers together."""
    return first_number * second_number

C:\GenerativeAIEgitim\Env1\Lib\site-packages\pydantic\_internal\_generate_schema.py:775: UserWarning: Mixing V1 models and V2 models (or constructs, like `TypeAdapter`) is not supported. Please upgrade `BaseMessage` to V2.
  warn(


In [10]:
llm_with_tools = llm.bind_tools([multiply])

In [11]:
agent_with_conditional_edges = MessageGraph()
#yepyeni başka bir agent yapalım

In [12]:
#şimdi şu node ları bu yeni yapı için baştan kurgulayalum

In [13]:
agent_with_conditional_edges.add_node("node1", llm_with_tools)
agent_with_conditional_edges.set_entry_point("node1")

In [14]:
tool_node = ToolNode([multiply])
agent_with_conditional_edges.add_node("multiply", tool_node)

In [15]:
agent_with_conditional_edges.add_edge("multiply", END)

In [16]:
#şimdi router ekliyoruz
#node1 den duruma göre node2 ye yönlenebilsin


In [17]:
from typing import Literal, List
from langchain_core.messages import BaseMessage

def router(state: List[BaseMessage]) -> Literal["multiply", "__end__"]:
    tool_calls = state[-1].additional_kwargs.get("tool_calls", [])
    if len(tool_calls):
        return "multiply"
    else:
        return "__end__"

#yukardaki kodun bantığı 
#node1 e bir tool çağırayımmı diye soruyor oradan ok gelirse multiply ı çağır diye dönüş yapıyor

agent_with_conditional_edges.add_conditional_edges("node1", router)

In [18]:
runnable_agent_with_conditional_edges = agent_with_conditional_edges.compile()

In [19]:
runnable_agent_with_conditional_edges.invoke(HumanMessage("What is 123 * 456?"))

[HumanMessage(content='What is 123 * 456?', id='364df9d1-4d78-453c-87c9-3bfb3354b771'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_iWRfZaJjHubfOoxzKDjrtMR3', 'function': {'arguments': '{"first_number":123,"second_number":456}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 57, 'total_tokens': 77, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_dbaca60df0', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-560835ed-0ddc-4c59-930c-07bf1695d47c-0', tool_calls=[{'name': 'multiply', 'args': {'first_number': 123, 'second_number': 456}, 'id': 'call_iWRfZaJjHubfOoxzKDjrtMR3', 'type': 'tool_call'}], usage_metadata={'input_tokens': 57, 'output_tokens': 20, 'total_toke

In [20]:
runnable_agent_with_conditional_edges.invoke(HumanMessage("Galatasaray ne zaman kuruldu?"))

[HumanMessage(content='Galatasaray ne zaman kuruldu?', id='5eb57f40-cb72-4c8d-9087-e6ba7e55c89a'),
 AIMessage(content="Galatasaray Spor Kulübü, 1 Ekim 1905 tarihinde İstanbul'da Galatasaray Lisesi'nde kuruldu.", response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 57, 'total_tokens': 89, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_dbaca60df0', 'finish_reason': 'stop', 'logprobs': None}, id='run-6da0b4ec-69f8-4205-ae96-a2a1bb2995c5-0', usage_metadata={'input_tokens': 57, 'output_tokens': 32, 'total_tokens': 89})]

In [21]:
# başka bir agent yapalım
#aynı işi yapan farklı bir yöntem

In [22]:
from langchain_community.tools.tavily_search import TavilySearchResults
online_search_tool = [TavilySearchResults(max_results=1)]

In [23]:
from langgraph.prebuilt import ToolNode
tool_node_with_online_search = ToolNode(online_search_tool)

In [24]:
llm_with_online_search_tool = llm.bind_tools(online_search_tool)

In [25]:
#şimdi stage ler ekleyeceğiz agent şu  an bu stage de 

In [26]:
from typing import TypedDict, Annotated

def add_messages(left: list, right: list):
    """Add-don't-overwrite."""
    return left + right

class AgentState(TypedDict):
    # The `add_messages` function within the annotation defines
    # *how* updates should be merged into the state.
    messages: Annotated[list, add_messages]

In [27]:
from typing import Literal

# Define the function that determines whether to continue or not
def should_continue(state: AgentState) -> Literal["node2", "__end__"]:
    messages = state['messages']
    last_message = messages[-1]
    # If the LLM makes a tool call, then we route to the "action" node
    if last_message.tool_calls:
        return "node2"
    # Otherwise, we stop (reply to the user)
    return "__end__"


# Define the function that calls the model
def call_model(state: AgentState):
    messages = state['messages']
    response = llm_with_online_search_tool.invoke(messages)    
    # We return a list, because this will get added to the existing list
    return {"messages": [response]}

In [28]:
from langgraph.graph import StateGraph, END
# Define a new graph
workflow = StateGraph(AgentState)

# Define the two nodes we will cycle between
workflow.add_node("node1", call_model)
workflow.add_node("node2", tool_node_with_online_search)

# Set the entrypoint as `node1`
# This means that this node is the first one called
workflow.set_entry_point("node1")

# We now add a conditional edge
workflow.add_conditional_edges(
    # First, we define the start node. We use `node1`.
    # This means these are the edges taken after the `node1` node is called.
    "node1",
    # Next, we pass in the function that will determine which node is called next.
    should_continue,
)

# We now add a normal edge from `online_search_tool` to `node1`.
# This means that after `online_search_tool` is called, `node1` node is called next.
workflow.add_edge('node2', 'node1')

In [29]:
agent_with_cycles = workflow.compile()

In [30]:
from langchain_core.messages import HumanMessage

inputs = {"messages": [HumanMessage(content="what is the weather in sf")]}
agent_with_cycles.invoke(inputs)

{'messages': [HumanMessage(content='what is the weather in sf'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_Z9DChckJIQH146h8xQFBYAyx', 'function': {'arguments': '{"query":"current weather in San Francisco"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 85, 'total_tokens': 108, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_dbaca60df0', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-217b59d8-f36c-405b-93ff-2702140d39a8-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'current weather in San Francisco'}, 'id': 'call_Z9DChckJIQH146h8xQFBYAyx', 'type': 'tool_call'}], usage_metadata={'input_tokens': 85, 'output_tok

In [31]:
inputs = {"messages": [HumanMessage(content="Merhaba nasılsın")]}
agent_with_cycles.invoke(inputs)

{'messages': [HumanMessage(content='Merhaba nasılsın'),
  AIMessage(content='Merhaba! Ben bir yapay zeka dil modeliyim, bu yüzden duygularım yok ama size yardımcı olmak için buradayım. Size nasıl yardımcı olabilirim?', response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 85, 'total_tokens': 123, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_dbaca60df0', 'finish_reason': 'stop', 'logprobs': None}, id='run-009aced7-ecb1-4e42-9c71-fc0d73fbd403-0', usage_metadata={'input_tokens': 85, 'output_tokens': 38, 'total_tokens': 123})]}

In [ ]:
"""
bu iki yapı arası fark
1. Router Agent (Agent with Conditional Edges)
Bu yapı daha şartlı dallanma (conditional branching) mantığına dayanıyor.

Amaç: Gelen mesajın içeriğine veya durumuna bakarak, hangi adımın (node) çalıştırılacağını dinamik olarak belirlemek.
Yapı:
Node1: LLM (Dil Modeli) üzerinden gelen mesajları işler.
Multiply Node: Eğer Node1’de bir tool çağrısı yapılırsa, bu tool kullanılarak işlemi gerçekleştirir.
Router Fonksiyonu: node1 tamamlandıktan sonra tool çağrısı yapılıp yapılmadığına bakar.
Tool çağrısı varsa multiply düğümüne gider.
Tool çağrısı yoksa işlemi sonlandırır (__end__).
Örnek Senaryolar:

"What is 123 * 456?": Bu durumda tool çağrısı gerektiği için multiply node’u çalışır ve çarpma işlemi yapılır.
"Galatasaray ne zaman kuruldu?": Tool çağrısı gerekmediği için doğrudan __end__ düğümüne gider ve işlem sonlanır.


2. Workflow Agent (Agent with StateGraph and Cycles)
Bu yapı daha workflow (akış) tabanlı ve döngüsel (cyclical) bir süreç izliyor.

Amaç: Agent’ın durumunu (state) takip etmek ve belirli koşullarda workflow’u döngüsel olarak tekrarlamak.
Yapı:
Node1: LLM’den gelen mesajı işler ve modelin yanıtını döndürür.
Node2 (Tool Node): Dış kaynak (örneğin Tavily Search gibi bir online arama aracı) ile bilgi arar.
StateGraph ve Koşullar:
Node1’den sonra gelen yanıtın içinde bir tool çağrısı varsa, node2 çalışır.
node2 tamamlandıktan sonra tekrar node1'e dönülür ve süreç tekrar eder.
Eğer tool çağrısı yoksa, işlem __end__ düğümüne giderek sonlanır.
Örnek Senaryolar:

"What is the weather in sf?": Node1’den yanıt geldikten sonra tool çağrısı tespit edilir ve node2 çalışarak online arama yapılır.
"Merhaba nasılsın": Tool çağrısı gerekmediği için işlem Node1’den sonra tamamlanır (__end__).

Sonuç:

Router Agent: Daha koşullu ve basit dallanmalar için kullanılır. Yalnızca belirli bir mantıkla düğümden düğüme geçer.
Workflow Agent: Daha karmaşık iş akışları ve döngüsel süreçler için tasarlanır. Agent’ın durumunu yönetmek ve farklı düğümler arasında tekrarlayan geçişler yapmak için idealdir.
"""